# Эмбеддинги и векторный поиск

Превращаем чанки из урока 3 в векторы фиксированной длины (эмбеддинги),
чтобы искать по документации по смыслу, а не по совпадению слов.

## Пересобираем чанки

Переменная `chunks` жила в `chunking.ipynb` — здесь собираем её заново
той же функцией `chunk_text()`.

In [3]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.

    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются

    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [4]:
from pathlib import Path

DOCS_DIR = Path("docs")

chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

Всего чанков: 530


## Считаем эмбеддинги для чанков

Модель `paraphrase-multilingual-MiniLM-L12-v2` — многоязычная (включая русский),
выдаёт векторы длиной 384. При первом запуске скачается с Hugging Face (~470 МБ),
дальше берётся из локального кеша.

`normalize_embeddings=True` приводит каждый вектор к единичной длине —
тогда косинусная близость эквивалентна скалярному произведению,
и с FAISS будет удобнее работать.

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)

# Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

Загружаем модель эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches: 100%|██████████| 17/17 [00:02<00:00,  5.69it/s]

Размер матрицы эмбеддингов: (530, 384)


## Заглянем внутрь

Сами числа ничего не говорят — они имеют смысл только в сравнении друг с другом.
Норма равна единице, потому что мы запросили нормализацию векторов.

In [4]:
print(f"Первый чанк: {chunks[0]['text'][:80]}...")
print(f"Его эмбеддинг (первые 10 чисел): {chunk_embeddings[0][:10]}")
print(f"Длина вектора (норма): {np.linalg.norm(chunk_embeddings[0]):.4f}")

Первый чанк: # Documentation

### Getting Started
* [Quickstart](https://docs.ollama.com/quic...
Его эмбеддинг (первые 10 чисел): [-0.0613074  -0.05510945 -0.03115823 -0.01005449  0.03433183  0.04836935
 -0.08265641  0.02022992 -0.00097118  0.04949648]
Длина вектора (норма): 1.0000
